<a href="https://colab.research.google.com/github/harshs-data/Pytorch/blob/main/Next_word_predictor_using_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [85]:
!pip install nltk


In [86]:
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from nltk.tokenize import word_tokenize
import nltk
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [87]:
document = """About the Program
What is the course fee for  Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program’s curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Checkout this google sheet to see month by month time table of the course - https://docs.google.com/spreadsheets/d/16OoTax_A6ORAeCg4emgexhqqPv3noQPYKU7RJ6ArOzk/edit?usp=sharing.
What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the link for our website - https://learnwith.campusx.in/
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription? Suppose if I pay on 15th Jan, then do I have to pay again on 1st Feb or 15th Feb
15th Feb. The validity period is 30 days from the day you make the payment. So essentially you can join anytime you don’t have to wait for a month to end.
What if I don’t like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
This one is tricky, so read carefully. You can watch the videos till your subscription is valid. Suppose you have purchased subscription on 21st Jan, you will be able to watch all the past paid sessions in the period of 21st Jan to 20th Feb. But after 21st Feb you will have to purchase the subscription again.
But once the course is over and you have paid us Rs 5600(or 7 installments of Rs 799) you will be able to watch the paid sessions till Aug 2024.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmai.com
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee. So we dont guarantee you any jobs or for that matter even interview calls. So if you are planning to join this course just for placements, I am afraid you will be disappointed. Here is what comes under placement assistance
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""


In [88]:
# tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [89]:
# tokenize
tokens = word_tokenize(document.lower())

In [90]:
# build vocab
vocab = {'<unk>': 0}

for token in Counter(tokens):
  if token not in vocab:
    vocab[token] = len(vocab)

In [91]:
len(vocab)

289

In [92]:
# extract sentences from data
input_sentences = document.split('\n')

In [93]:
def text_to_indeces(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence

In [94]:
input_numerical_sentences = []

for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indeces(word_tokenize(sentence.lower()), vocab))

In [95]:
len(input_numerical_sentences)

78

In [96]:
training_sequence = []

for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])


In [97]:
training_sequence

[[1, 2],
 [1, 2, 3],
 [4, 5],
 [4, 5, 2],
 [4, 5, 2, 6],
 [4, 5, 2, 6, 7],
 [4, 5, 2, 6, 7, 8],
 [4, 5, 2, 6, 7, 8, 9],
 [4, 5, 2, 6, 7, 8, 9, 10],
 [4, 5, 2, 6, 7, 8, 9, 10, 11],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12, 13],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12, 13, 14],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12, 13, 14, 15],
 [2, 6],
 [2, 6, 16],
 [2, 6, 16, 17],
 [2, 6, 16, 17, 18],
 [2, 6, 16, 17, 18, 19],
 [2, 6, 16, 17, 18, 19, 20],
 [2, 6, 16, 17, 18, 19, 20, 21],
 [2, 6, 16, 17, 18, 19, 20, 21, 22],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18, 26],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18, 26, 27],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18, 26, 27, 28],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18

In [98]:
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max(len_list)

62

In [99]:
padded_training_sequence = []

for sequence in training_sequence:
  padded_training_sequence.append([0]*(max(len_list)-len(sequence)) + sequence)

In [100]:
print(len(padded_training_sequence[4]))
print(len(padded_training_sequence[1]))

62
62


In [101]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype = torch.long)

In [102]:
padded_training_sequence.shape

torch.Size([942, 62])

In [103]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[: , -1]

In [104]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]

In [105]:
dataset = CustomDataset(X,y)

In [106]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [107]:
for input, output in dataloader:
  print(input,output)


tensor([[  0,   0,   0,  ..., 146,  24, 107],
        [  0,   0,   0,  ...,   0,   0,   4],
        [  0,   0,   0,  ...,  46, 146,  24],
        ...,
        [  0,   0,   0,  ..., 165, 166, 167],
        [  0,   0,   0,  ...,  93, 151,  18],
        [  0,   0,   0,  ..., 182,  77,  78]]) tensor([ 93,   5, 100,  12, 223, 154,  95,  73,  29, 183,  69,  30,  85,  80,
         77, 180,  17, 152,  52,  33, 121, 225,  86,   2, 238,  89,  23,  33,
         81, 250,  26,   2])
tensor([[  0,   0,   0,  ...,   0,  22,  23],
        [  0,   0,   0,  ...,  30,  68,   5],
        [  0,   0,   0,  ...,   0,  21,  65],
        ...,
        [  0,   0,   0,  ...,  65, 141, 144],
        [  0,   0,   0,  ...,  78,  36, 219],
        [  0,   0,   0,  ...,  24,  90,   2]]) tensor([ 24,   4,  86,   3,   9, 176,   4, 155,  33,  22,  24, 194,  23, 165,
         17,   2,  16, 208, 223, 158,   2,  78, 127, 110,  97, 200,   2,   5,
        174,  22, 220, 251])
tensor([[  0,   0,   0,  ...,   0,   0,  21],
    

In [108]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [109]:
model = LSTMModel(len(vocab))


In [110]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
print('-'*20)
model.to(device)

cuda
--------------------


LSTMModel(
  (embedding): Embedding(289, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=289, bias=True)
)

In [111]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr= learning_rate)

In [112]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_X, batch_y in dataloader:
    batch_X, batch_y = batch_X.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_X)

    loss = criterion(output, batch_y)
    loss.backward()
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch +1}, loss: {total_loss: .4f}")


Epoch: 1, loss:  165.4638
Epoch: 2, loss:  146.4682
Epoch: 3, loss:  133.9755
Epoch: 4, loss:  121.6331
Epoch: 5, loss:  109.9863
Epoch: 6, loss:  98.1733
Epoch: 7, loss:  87.2902
Epoch: 8, loss:  78.0996
Epoch: 9, loss:  69.2349
Epoch: 10, loss:  60.8498
Epoch: 11, loss:  54.1005
Epoch: 12, loss:  47.6224
Epoch: 13, loss:  42.0188
Epoch: 14, loss:  36.5235
Epoch: 15, loss:  32.2678
Epoch: 16, loss:  28.4577
Epoch: 17, loss:  25.1193
Epoch: 18, loss:  22.1116
Epoch: 19, loss:  19.8330
Epoch: 20, loss:  17.6713
Epoch: 21, loss:  15.8125
Epoch: 22, loss:  14.3648
Epoch: 23, loss:  13.2712
Epoch: 24, loss:  12.1609
Epoch: 25, loss:  11.2398
Epoch: 26, loss:  10.0812
Epoch: 27, loss:  9.5945
Epoch: 28, loss:  8.8030
Epoch: 29, loss:  8.3079
Epoch: 30, loss:  7.8931
Epoch: 31, loss:  7.5049
Epoch: 32, loss:  7.1329
Epoch: 33, loss:  6.7554
Epoch: 34, loss:  6.5150
Epoch: 35, loss:  6.3286
Epoch: 36, loss:  6.0702
Epoch: 37, loss:  5.8954
Epoch: 38, loss:  5.6854
Epoch: 39, loss:  5.5559
Epo

In [154]:
def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # convert to numerical
  numerical_text = text_to_indeces(tokenized_text, vocab)

  # padding
  padded_text = torch.tensor([0]*(61 -len(numerical_text)) + numerical_text, dtype = torch.long).unsqueeze(0)

  # send to model
  padded_text = padded_text.to(device) # Move tensor to the same device as the model
  output = model(padded_text)


  # predicted index
  value, index = torch.max(output, dim = 1)

  # merge with text
  return text + ' ' + list(vocab.keys())[index]

In [155]:
prediction(model, vocab, "The course follows")

'The course follows a'

In [156]:
num_tokens = 10
input_text = "The course follows"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text


The course follows a
The course follows a monthly
The course follows a monthly subscription
The course follows a monthly subscription model
The course follows a monthly subscription model where
The course follows a monthly subscription model where you
The course follows a monthly subscription model where you have
The course follows a monthly subscription model where you have to
The course follows a monthly subscription model where you have to make
The course follows a monthly subscription model where you have to make monthly
